*0.4 Deep learning basics*

# autograd

**The situation.** In 0.2 gradient descent found "12 ms per token" by walking downhill on a loss. Nobody wrote the derivative. A model has millions of parameters and a loss built from hundreds of operations; writing derivatives by hand is impossible, and it is what stopped neural networks for decades.

**Autograd.** PyTorch records every operation on tensors that have `requires_grad=True`, building a graph. `loss.backward()` walks the graph backwards and fills `.grad` on every parameter with the gradient — how much the loss changes per unit change of that parameter. Any loss you can write, it can differentiate.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch

# A tiny cost model: latency = base + per_token × tokens. Two parameters to learn.
base = torch.tensor(200.0, requires_grad=True)
per_token = torch.tensor(5.0, requires_grad=True)
tokens = torch.tensor([100.0, 300.0, 500.0])
observed_ms = torch.tensor([1500.0, 3900.0, 6300.0])  # truth: 300 + 12 × tokens

predicted = base + per_token * tokens
loss = ((predicted - observed_ms) ** 2).mean()
loss.backward()  # fill .grad on base and per_token

print("loss:", round(loss.item(), 1))
print("d loss / d base:     ", round(base.grad.item(), 1))
print("d loss / d per_token:", round(per_token.grad.item(), 1))

# Check against the derivative written by hand: d/d base of mean((pred - y)^2) = 2 × mean(pred - y)
by_hand = 2 * (predicted - observed_ms).mean()
print("by hand, d loss / d base:", round(by_hand.item(), 1))
assert torch.isclose(base.grad, by_hand)

loss: 6146666.5
d loss / d base:      -4400.0
d loss / d per_token: -1693333.4
by hand, d loss / d base: -4400.0


**Reading the output.** Both gradients are negative and large: the loss falls if `base` and `per_token` go up — correct, since the guesses (200, 5) are below the truth (300, 12). The hand-computed derivative matches autograd to the decimal. For a real model the same call fills millions of `.grad` values.

**Two rules of use.** Gradients accumulate across calls, so zero them each step; and at inference, switch recording off — it costs memory and time.

In [3]:
base.grad.zero_()
per_token.grad.zero_()
for _ in range(2):
    ((base + per_token * tokens - observed_ms) ** 2).mean().backward()
print(
    "after two backward() calls without zeroing:",
    round(base.grad.item(), 1),
    "(doubled — accumulated)",
)

with torch.no_grad():  # inference: no graph, no memory for it
    inference = base + per_token * tokens
print("inside no_grad, requires_grad:", inference.requires_grad)
assert not inference.requires_grad and abs(base.grad.item() - 2 * by_hand.item()) < 1

after two backward() calls without zeroing: -8800.0 (doubled — accumulated)
inside no_grad, requires_grad: False


```
forward   base ──┐
                 ├─ predicted ── (−observed)² ── mean ── loss
   per_token ────┘                                        │
backward  base.grad ◀────────────────────────────────────┘  chain rule, automatically
```

**The rule to remember.** Write the forward computation; autograd writes the backward. Zero grads every step, `no_grad` at inference.

| Use it when | Don't when | Instead use |
|---|---|---|
| training anything; sensitivity analysis | inference, evaluation, data preprocessing | `torch.no_grad()` / `torch.inference_mode()` |

**Watch out**
- `.grad` accumulates. Forgetting `zero_grad()` is the most common silent training bug.
- Operations that break the graph (`.item()`, `.numpy()`, `.detach()`) stop gradients flowing through them — sometimes on purpose, often by accident.
- Keeping references to `loss` tensors across steps keeps their graphs alive: a memory leak that looks like a GPU that "fills up over time".